# A humanoid at the ping-pong table

This notebook puts a Unitree G1 with HIM's red paddle at the HIM table, in Isaac Lab, with the calibrated ball physics.
It runs in two parts:

1. **The template scene**: you step the robot with your own actions.
2. **The trained policy**: the G1 rallies against HIM's opponent.

The ball is not a PhysX body. It is the calibrated ball physics, flown in 1 ms sub-steps against the robot's real
paddle every 5 ms physics step, and pushing back on it.

Open it from a session on the `robot` service:

```bash
antioch session new --profile robot
antioch jupyter lab
```

## 1. Start Isaac Lab

Start Isaac Lab once per kernel. Each section builds its scene, then closes it before the next.

In [ ]:
import os

import antioch
import torch

from lib.config import ROBOT_SIMULATION_CONFIG
from lib.robots import UNITREE_G1
from lib.robots.table_scene import build_table_scene

if os.environ.get("ANTIOCH_ENGINE") != "isaac-lab-3.0":
    raise RuntimeError("open this notebook from `antioch session new --profile robot`")
antioch.start_simulation(ROBOT_SIMULATION_CONFIG)

## 2. The template scene

`build_table_scene` builds the robot from its spec (`lib/robots/unitree_g1.py`) in parallel worlds. Each world has:

- **the robot:** its USD with the hand pruned and the paddle fixed to its wrist, every link colliding as its URDF
  says, and its motors and PD gains;
- **the table and net**, as the robot meets them;
- **the calibrated ball.**

Here the pelvis is pinned (`fixed_base=True`): with no balance controller, a G1 holding its pose would fall.
Set `fixed_base=False` once your controller balances; a fallen robot is then reset to its stance.

In [ ]:
scene = build_table_scene(UNITREE_G1, num_envs=4, fixed_base=True, reset_spread=False)
observation = scene.reset_all()
print(f"{scene.num_envs} envs, {len(scene.joint_names)} joints, observation {tuple(observation.shape)}, policy step {scene.step_dt} s")
scene.joint_names[:6]

## 3. Step your own actions

An action is one number per joint. The joint's target is `default_pose + action_scale * action`, and its PD gains
drive it there; the scale is a quarter of the joint's peak torque over its stiffness. Below, the right arm swings while
one of HIM's recorded feeds comes in. Replace `my_controller` with yours. The observation's columns are listed by
`observation_terms` in `lib/robots/table_scene.py`.

In [ ]:
from lib.robots.template import RECORDED_FEEDS

swing = scene.joint_names.index("right_shoulder_pitch_joint")


def my_controller(observation: torch.Tensor, t: float) -> torch.Tensor:
    actions = scene.hold_default_pose()
    actions[:, swing] = 3.0 * torch.sin(torch.tensor(2.0 * torch.pi * t))  # about 0.5 rad either way
    return actions


coupling = scene.coupling
rows = torch.arange(scene.num_envs, device=coupling.device)
coupling.launch(rows, torch.tensor(RECORDED_FEEDS[0], dtype=coupling.dtype, device=coupling.device).expand(scene.num_envs, 9))
for step in range(100):  # 2 s
    observation = scene.step(my_controller(observation, step * scene.step_dt))
ball = coupling.ball[0].cpu()
print(f"after 2 s: ball in play {bool(coupling.active[0])}, at {ball[:3].numpy().round(3)} m (table frame)")
print(f"robot-blade contacts so far: {coupling.counters.contacts}")

In [ ]:
scene.close()

## 4. The trained policy

The trained G1 policy rallies against HIM's distilled opponent; `docs/policy.md` covers it.

- **The actor:** `load_rally_actor` fetches the checkpoint and folds it to the 146 inputs it reads at play.
- **The rallies:** `CoupledRally` is the scene's hooks. It hands the coupling the opponent's paddle every physics step,
  opens rallies with the opponent's recorded feeds, and adds the ball forecast to the observation.

Ten seconds at 64 envs:

In [ ]:
from lib.assets import G1_RALLY_POLICY
from lib.policy.actor import hide_ball, load_rally_actor
from lib.policy.grade import rally_metrics
from lib.policy.rally import CoupledRally
from lib.policy.run import PUSH_SPEED_MPS

rally = CoupledRally(seed=0, forecast_through_serve=True)
scene = build_table_scene(UNITREE_G1, 64, hooks=rally, opponent=True, push_speed_mps=PUSH_SPEED_MPS)
actor = load_rally_actor(G1_RALLY_POLICY, default_pose=UNITREE_G1.default_pose, device=scene.env.device)
played = rally_metrics(rally.play(scene, actor, 10.0))
{key: played[key] for key in ("receives", "legal_per_receive", "legal_per_receive_ci95", "hit_rate", "rally_length_legal_shots", "falls_per_10s")}

## 5. The blind control

The same weights, with the ball and its forecast hidden: a policy that plays from the ball must collapse.

In [ ]:
blind = rally_metrics(rally.play(scene, hide_ball(actor), 10.0))
{key: blind[key] for key in ("receives", "legal_per_receive", "hit_rate")}

In [ ]:
scene.close()

## 6. The recorded scenarios

The same features run as recorded scenario runs, with their checks, telemetry and the 24 fps film of env 0 in the HIM
lab twin:

```bash
uv run antioch scenario run load_humanoid_at_table --case unitree-g1
uv run antioch scenario run play_ping_pong_policy --case g1-rally-smoke
```

Close Isaac when you are done, then release the session with `antioch session release`.

In [ ]:
antioch.application().close()